In [ ]:
import pandas as pd

df = pd.read_csv("trips.csv")

In [ ]:
# Lab 1 — Basic Aggregation on One Column

mean_fare = df["fare"].mean()
median_fare = df["fare"].median()
min_fare = df["fare"].min()
max_fare = df["fare"].max()
std_fare = df["fare"].std()
count_fare = df["fare"].count()

print("Mean:", mean_fare)
print("Median:", median_fare)
print("Minimum:", min_fare)
print("Maximum:", max_fare)
print("Standard deviation:", std_fare)
print("Count:", count_fare)



Interpretation:
The average fare is about $20.30, while the median fare is $17.31. The mean is higher than the median, suggesting that some relatively expensive trips may pull the mean upward.

In [ ]:
# Lab 2 — Multiple Aggregations with agg()

wait_summary = df["waiting_time_min"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max"
])

print(wait_summary)

Interpretation:
The mean waiting time is 5.18 minutes, while the median is 4.80 minutes. Since the mean is higher than the median and the maximum is 39.5 minutes, the distribution appears to have some unusually long waiting times and is likely right-skewed.

In [1]:
# Lab 3 — Average Waiting Time by Zone

zone_wait = (
    df.groupby("zone")["waiting_time_min"]
      .mean()
      .sort_values(ascending=False)
)

print(zone_wait)

NameError: name 'df' is not defined

Airport has the highest average waiting time, approximately 6.10 minutes.

Downtown has the lowest average waiting time, approximately 4.64 minutes.

In [ ]:
# Lab 4 — Multiple Statistics by Group

zone_summary = df.groupby("zone")["waiting_time_min"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

print(zone_summary)

Largest number of valid waiting-time observations: Downtown, 105.

Greatest variability: University, because:
        
which is much larger than the other zones.

Highest mean: Airport, 6.10.

Highest median: Airport, 5.90.

University is particularly interesting:
Mean   = 5.55
Median = 4.20
Max    = 39.50

This suggests that one or more unusually large waiting times are pulling the mean upward.

Why report center and variability?
Two groups can have similar average values but very different levels of consistency. The mean or median describes the center, while the standard deviation and range describe how spread out the observations are.

In [3]:
# Lab 5 — Aggregate Multiple Numerical Variables

zone_means = df.groupby("zone")[
    ["distance_km", "fare", "waiting_time_min"]
].mean()

print(zone_means)

NameError: name 'df' is not defined

Longest average distance: Industrial, 8.44 km.

Highest average fare: Airport, $30.73.

Highest average waiting time: Airport, 6.10 minutes.

In [ ]:
# Lab 6 — Named Aggregation

zone_summary = df.groupby("zone").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    median_fare=("fare", "median"),
    avg_wait=("waiting_time_min", "mean"),
    max_wait=("waiting_time_min", "max")
)

zone_summary = zone_summary.sort_values(
    "avg_wait",
    ascending=False
)

print(zone_summary)

Airport has the highest average waiting time.

In [ ]:
# Lab 7 — Group by Two Variables

zone_weather = (
    df.groupby(["zone", "weather"])["waiting_time_min"]
      .agg(["count", "mean", "median"])
)

print(zone_weather)


Highest mean:
Airport + Snow = 9.90 minutes

In [ ]:
# Lab 8 — Peak vs Off-Peak
def get_period(hour):
    if hour in [7, 8, 9, 16, 17, 18]:
        return "Peak"
    else:
        return "Off-Peak"

df["period"] = df["hour"].apply(get_period)


period_summary = df.groupby("period").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    avg_wait=("waiting_time_min", "mean"),
    median_wait=("waiting_time_min", "median"),
    avg_demand=("demand", "mean"),
    avg_supply=("driver_supply", "mean")
)

print(period_summary)

Both demand and driver supply are higher during peak periods. However, demand increases more strongly than supply, and waiting times are slightly higher during peak periods.

In [ ]:
# Lab 9 — Custom Aggregation Function

# Function：
def data_range(x):
    return x.max() - x.min()

# Overall fare range：
df["fare"].agg(data_range)

# By zone：
df.groupby("zone")["fare"].agg(data_range)

# Waiting-time range：
df.groupby("zone")["waiting_time_min"].agg(data_range)

University has the largest waiting-time range: 38.4 minutes.

In [ ]:
# Lab 10 — Aggregation After Filtering

# Filter：

longer_waits = df[
    df["waiting_time_min"] > 5
]

# Aggregation：

filtered_summary = longer_waits.groupby("zone")[
    "waiting_time_min"
].agg(["count", "mean"])

print(filtered_summary)



Filtering changes the population being summarized. Therefore, an average calculated after filtering answers a different question from the average calculated using the full dataset.

In [ ]:
# Lab 11 — groupby() + sort_values()

zone_rank = df.groupby("zone").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    avg_wait=("waiting_time_min", "mean")
)

zone_rank = zone_rank.sort_values(
    "avg_wait",
    ascending=False
)

print(zone_rank)

# Top 3:
zone_rank.head(3)

# Sort by trip count：
zone_rank.sort_values(
    "trip_count",
    ascending=False
)


Rankings depend on the metric being used. “Busiest” and “slowest service” are different concepts.

In [5]:
# Lab 12 — Pivot Table
# Mean:
    
pivot_mean = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc="mean"
)

print(pivot_mean)

# Overall weather means：
df.groupby("weather")["waiting_time_min"].mean()


# Median pivot：
pivot_median = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc="median"
)


NameError: name 'pd' is not defined

In [6]:
# Lab 13 — Pivot Table with Mean and Median

pivot_both = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc=["mean", "median"]
)

print(pivot_both)

SyntaxError: invalid character '—' (U+2014) (3525810739.py, line 1)

In [ ]:
# Lab 14 — Supply and Demand

df["supply_demand_gap"] = (
    df["driver_supply"] - df["demand"]
)


supply_summary = df.groupby("zone").agg(
    avg_demand=("demand", "mean"),
    avg_supply=("driver_supply", "mean"),
    avg_gap=("supply_demand_gap", "mean"),
    avg_wait=("waiting_time_min", "mean")
)

print(supply_summary)

